# 内部表現実験の追加検証（実行用）

Colabの「ランタイム → ランタイムのタイプを変更 → T4 GPU」を選択。

- v2: 192 → **1,152入力**（指示3種類 × 文脈書式2種類）
- v3: 288 → **3,456入力**（手がかり72文追加で2倍 × 指示3種類 × 書式2種類）
- 各入力は全24層を測定し、Probeの学習構成を20通りで比較
- 学習ラベルを説明文単位でシャッフルする負の対照も実施
- 独立した概念ペア数は **8のまま**。入力数・Probe数をサンプル数として検定しない
- 元の説明文評価は小さく、新規手がかりは人手評価・厳密な長さ合わせ前。探索的な頑健性確認

同じ入力の繰り返しではなく、入力・学習データを変える検証。新しいモデル推論はこのノートブックを実行して初めて得られる。
自動GitHubアップロードなし。元の実験結果は上書きしない。


In [ ]:
# Install in the disposable Colab runtime. Keep its CUDA-compatible torch.
%pip -q uninstall -y torchaudio torchvision
%pip -q install "transformers==4.57.6" "scikit-learn>=1.3,<2" "scipy>=1.11" "pandas>=2.1" "numpy>=1.26" "tqdm>=4.65" "requests>=2.31"


In [ ]:
from pathlib import Path
import subprocess, sys
REPO = Path('/content/GraduationThesis-robustness-20261103')
if not (REPO / '.git').exists():
    subprocess.run(['git','clone','--depth','1','--branch','main',
                    'https://github.com/ryouy/GraduationThesis.git',str(REPO)],check=True)
print('Source commit:', subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
subprocess.run([sys.executable,'-m','unittest','discover','-s','experiments/2026-11/1103_review','-p','test_*.py','-v'],cwd=REPO,check=True)


In [ ]:
# Validate the planned inputs without running a model.
subprocess.run([sys.executable,'experiments/2026-11/1103_review/run_robustness.py','--dry-run',
                '--out','/content/representation-design-only'],cwd=REPO,check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), "ランタイムのタイプをT4 GPUに変更してください"
OUT = Path('/content/representation-robustness-run')
OUT.mkdir(parents=True, exist_ok=True)
# Preserve stdout/stderr even when Colab's rich error renderer is unavailable.
with (OUT/'execution.log').open('a', encoding='utf-8') as execution_log:
    process = subprocess.Popen([sys.executable,'-u','experiments/2026-11/1103_review/run_robustness.py',
                                '--out',str(OUT)], cwd=REPO,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        execution_log.write(line)
        execution_log.flush()
        print(line, end='', flush=True)
    return_code = process.wait()
assert return_code == 0, f"実行失敗: execution.log を確認してください (exit={return_code})"


In [ ]:
import json, pandas as pd
assert (OUT/'completed_manifest.json').exists(), 'Inference/analysis has not completed.'
print((OUT/'completed_manifest.json').read_text())
v3 = pd.read_csv(OUT/'v3_focus_sensitivity.csv')
display(v3.query('probe_variant == "original"'))
v2 = pd.read_csv(OUT/'v2_distance_sensitivity.csv')
display(v2.query('probe_variant == "original" and layer in [22,23,24]'))
display(pd.read_csv(OUT/'instruction_length_audit.csv'))
display(pd.read_csv(OUT/'probe_coverage.csv').query('layer in [22,23,24]'))


## 結果の判断

元の手がかり／新規手がかり、X／Y、指示3種類、文脈書式2種類を分けて見る。Probeを変えたときの符号・AUCの安定性と、シャッフルラベルのAUCを確認する。選んだ条件だけを取り上げて結論にしない。

v2の22/23層は既存データを見て注目した層であり、独立した事前登録検証ではない。24層・多条件を眺めた結果の信頼区間を、確証的な有意差として扱わない。内部の絶対活性、忘却、因果的な抑制回路はこの実験では直接測れない。

下のZIPには再解析用Hidden Stateキャッシュも含まれるため、サイズが大きくなる。ダウンロードした結果を保存してからランタイムを終了する。


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/representation-robustness-results','zip',root_dir=OUT)
files.download(archive)
